# 第 04 课：文本文件与 JSON

**前置知识：** 字符串、列表、字典、循环、条件判断，以及第 03 课的函数和返回值。  
**建议用时：** 90–120 分钟，可分两次完成。  
**配套脚本：** `lesson_04_text_files_and_json.py`。

今天完成一个完整流程：从文本文件读取多行文字，逐行调用 encode，把结果保存为 JSON，再从文件读回来核对。

本课自带 encode/decode 的完整定义和示例输入生成代码。下载这份讲义或配套脚本即可开始，不需要额外数据文件。

本课只处理很小的练习文件，一次把全文读入内存。以后处理大型数据集时，再学习逐行读取。


## 使用方式与文件位置

选择 Python 3 内核，从上到下按 Shift + Enter 运行。做到练习区先停下，独立完成后再看参考答案。

本课会在**当前工作目录**下建立 `lesson_04_data` 文件夹：

| 文件 | 用途 | 重复运行时的行为 |
| --- | --- | --- |
| `input.txt` | 供你编辑的输入文字 | 缺失时生成示例；已存在时读取现有内容 |
| `generated_records.json` | 完整程序生成的编码结果 | 根据当前输入重新写入 |
| `exercise_b_records.json` | 练习 B 参考答案的输出 | 执行该参考答案时重新写入 |
| `exercise_c_records.json` | 练习 C 参考答案的输出 | 执行该参考答案时重新写入 |

“当前工作目录”取决于你启动程序的位置，不一定等于 `.py` 文件所在目录。程序会打印实际路径。

已保留的输出来自验证环境；你重新运行后，路径会变成你自己电脑上的位置。练习区只有合法起始代码和注释，未完成练习也不会阻断“运行全部单元”。


## 1. 导入 Python 自带的工具

`import json` 让程序使用 JSON 转换功能；`from pathlib import Path` 取出处理文件路径的 Path 工具。

它们都是 Python 标准库的一部分。本课通过 import 使用它们，不需要另外安装。


In [1]:
import json
from pathlib import Path


## 2. 用 Path 表示文件位置

- `Path.cwd()` 取得当前工作目录。
- 对 Path 使用 `/ "子目录或文件名"`，可以拼接路径。
- `mkdir(exist_ok=True)` 创建目录；目录已经存在时，也可以继续执行。
- `resolve()` 在这里用来显示实际的绝对路径。

`exist_ok=True` 是按参数名传入设置的写法。后面的 `encoding="utf-8"` 也采用同样的形式。


In [2]:
work_dir = Path.cwd() / "lesson_04_data"
work_dir.mkdir(exist_ok=True)
input_path = work_dir / "input.txt"
output_path = work_dir / "generated_records.json"

print("本课数据目录：", work_dir.resolve())


本课数据目录： /workspace/scratch/74d485b2a853/.notebook_build/lesson04_verify_8wzukpot/notebook run/lesson_04_data


## 3. 首次创建输入文件

`write_text(...)` 把字符串写入指定文件；`encoding="utf-8"` 指定保存文字使用的编码。

字符串中的 `\n` 表示换行。示例文件有下面四行，其中第 3 行为空行：

```text
我爱学习
我爱大模型

学习模型
```

`input_path.exists()` 判断路径是否存在。`not` 把判断结果反过来，所以 `if not input_path.exists():` 表示“文件还不存在时才执行下面的写入”。

这个判断让你在修改 input.txt 后，再次运行程序仍然使用修改后的输入。


In [3]:
# 只有输入文件不存在时，才创建示例；已存在的输入会被保留。
sample_text = '我爱学习\n我爱大模型\n\n学习模型\n'
if not input_path.exists():
    input_path.write_text(sample_text, encoding="utf-8")

print("输入文件：", input_path.resolve())


输入文件： /workspace/scratch/74d485b2a853/.notebook_build/lesson04_verify_8wzukpot/notebook run/lesson_04_data/input.txt


## 4. 读取文字，再按行拆分

`read_text(encoding="utf-8")` 返回一个 Python 字符串。

`splitlines()` 把字符串拆成每行一个元素的列表。本课保留行内空格与中间空行；默认示例的空行会得到空字符串。

`len(lines)` 表示列表里有多少个元素，也就是这里处理的行数。

**先预测：** 默认示例拆分后有几行？空行会变成什么？


In [4]:
raw_text = input_path.read_text(encoding="utf-8")
lines = raw_text.splitlines()

print("拆分后的列表：", lines)
print("行数：", len(lines))


拆分后的列表： ['我爱学习', '我爱大模型', '', '学习模型']
行数： 4


默认输出应是：

```python
["我爱学习", "我爱大模型", "", "学习模型"]
```

末尾的单个换行表示最后一行结束，不会额外产生一个空元素。完全空的文件会得到 `[]`；包含一个空行的文件则会得到 `[""]`。

输入中的空格仍保留在每行文字中。如果空格不在我们的字符词表里，encode 会按上一课的规则把它编码为未知字符。


## 5. 沿用上一课的函数

下一单元包含 encode、decode 和两份对应词表。运行后，本课就能独立使用这些函数。

编号仍保持一致：“我、爱、学、习、模、型”对应 0–5，未知标记对应 6。


In [5]:
def encode(text, token_to_id):
    token_ids = []

    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]

        token_ids.append(token_id)

    return token_ids

def decode(token_ids, id_to_token):
    restored = ""

    for token_id in token_ids:
        restored = restored + id_to_token[token_id]

    return restored

token_to_id = {
    "我": 0,
    "爱": 1,
    "学": 2,
    "习": 3,
    "模": 4,
    "型": 5,
    "<unk>": 6,
}
id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]


## 6. 每行文字对应一条记录

用一个字典保存一行的处理结果，再把多个字典放进 records 列表：

```python
{
    "line_number": 1,
    "text": "我爱学习",
    "token_ids": [0, 1, 2, 3]
}
```

line_number 是方便回查原文件的行号，从 1 开始；它与从 0 开始的字符编号是两种不同用途的数字。

循环每处理一行，就调用 encode、创建字典并追加到列表。行号在最后加 1。


In [6]:
records = []
line_number = 1

for text in lines:
    token_ids = encode(text, token_to_id)
    record = {
        "line_number": line_number,
        "text": text,
        "token_ids": token_ids,
    }
    records.append(record)
    line_number = line_number + 1

print("记录数量：", len(records))
for record in records:
    print(record)


记录数量： 4
{'line_number': 1, 'text': '我爱学习', 'token_ids': [0, 1, 2, 3]}
{'line_number': 2, 'text': '我爱大模型', 'token_ids': [0, 1, 6, 4, 5]}
{'line_number': 3, 'text': '', 'token_ids': []}
{'line_number': 4, 'text': '学习模型', 'token_ids': [2, 3, 4, 5]}


## 7. Python 数据与 JSON 文本

records 是内存中的 Python 列表。JSON 是一种文本数据格式，可以把列表、字典、字符串和数字等信息保存成文件。

今天先记住两个方向：

| 操作 | 输入 | 返回结果 |
| --- | --- | --- |
| `json.dumps(...)` | Python 列表或字典等数据 | JSON 格式的字符串 |
| `json.loads(...)` | JSON 格式的字符串 | 解析得到的 Python 数据 |

把 Python 数据转换成 JSON 文本称为“序列化”。它是在保存数据的结构；前面的 encode 则是在给字符查编号。

先用一条记录观察这两个方向。


In [7]:
one_record = {"text": "我爱", "token_ids": [0, 1]}
one_json_text = json.dumps(one_record, ensure_ascii=False, indent=2)
print(one_json_text)


{
  "text": "我爱",
  "token_ids": [
    0,
    1
  ]
}


- `ensure_ascii=False` 让 JSON 文本中的中文直接显示，便于阅读。
- `indent=2` 使用两个空格缩进，让保存后的结构更清楚。

这两个设置控制 JSON 文本的表示方式。文件使用什么文字编码，则由 read_text/write_text 中的 `encoding="utf-8"` 指定。

现在把刚才的 JSON 字符串解析回来，再按字典的键读取内容。


In [8]:
restored_record = json.loads(one_json_text)
print(restored_record["text"])
print(restored_record["token_ids"])


我爱
[0, 1]


## 8. 把全部记录保存到 JSON 文件

先让 dumps 返回 JSON 字符串，再把字符串写入文件。运行后，可按打印出的路径打开 generated_records.json 查看。

这是生成结果文件：每次执行保存步骤，都用当前 records 重新写入。


In [9]:
# 这是重新生成的结果文件，每次运行都会按当前输入更新。
json_text = json.dumps(records, ensure_ascii=False, indent=2)
output_path.write_text(json_text, encoding="utf-8")

print("已保存：", output_path.resolve())


已保存： /workspace/scratch/74d485b2a853/.notebook_build/lesson04_verify_8wzukpot/notebook run/lesson_04_data/generated_records.json


## 9. 重新读取保存结果

这里明确重新读取文件中的文字，再用 loads 解析它。

loaded_records 是一个新读入的列表。循环中的 record 是一条字典记录，可以用 `record["text"]` 和 `record["token_ids"]` 取出对应字段。

我们还会把读回来的编号交给 decode，检查字符处理结果。


In [10]:
saved_text = output_path.read_text(encoding="utf-8")
loaded_records = json.loads(saved_text)

print("工作目录：", work_dir.resolve())
print("输入文件：", input_path.resolve())
print("结果文件：", output_path.resolve())
print("记录数量：", len(loaded_records))

for record in loaded_records:
    restored = decode(record["token_ids"], id_to_token)
    print("行号：", record["line_number"])
    print("原文：", record["text"])
    print("编号：", record["token_ids"])
    print("解码：", restored)
    print()


工作目录： /workspace/scratch/74d485b2a853/.notebook_build/lesson04_verify_8wzukpot/notebook run/lesson_04_data
输入文件： /workspace/scratch/74d485b2a853/.notebook_build/lesson04_verify_8wzukpot/notebook run/lesson_04_data/input.txt
结果文件： /workspace/scratch/74d485b2a853/.notebook_build/lesson04_verify_8wzukpot/notebook run/lesson_04_data/generated_records.json
记录数量： 4
行号： 1
原文： 我爱学习
编号： [0, 1, 2, 3]
解码： 我爱学习

行号： 2
原文： 我爱大模型
编号： [0, 1, 6, 4, 5]
解码： 我爱<unk>模型

行号： 3
原文： 
编号： []
解码： 

行号： 4
原文： 学习模型
编号： [2, 3, 4, 5]
解码： 学习模型



默认输入会产生 4 条记录，包括空行对应的空编号列表。

第 2 行里，“大”的编号仍然是 6，所以仅凭 token_ids 解码只能得到“我爱<unk>模型”。JSON 中还单独保存了原文 text 字段，因此你也能从这个字段读到完整原文。它们保存的信息不同。

## 10. 完整程序

下一单元与配套 `.py` 文件内容一致，包含全部导入、函数、词表和文件处理步骤，可以独立运行。

在脚本所在文件夹打开终端，执行 `python lesson_04_text_files_and_json.py`。如果此前使用 py 或 python3，沿用对应命令。

完整程序会打印实际输入和输出路径。第一次运行生成示例输入，以后直接读取现有 input.txt。


In [11]:
# 第 04 课：读取文本文件，保存 JSON，再读取核对。
# 运行方法：python lesson_04_text_files_and_json.py
# 只使用 Python 标准库；示例输入会在首次运行时自动生成。
# 数据目录位于运行时的当前工作目录下，程序会打印实际位置。
# input.txt 已存在时会保留；generated_records.json 是每次更新的生成结果。
# 本课按行保留文字、空格与中间空行；空文件会产生空记录列表。

import json
from pathlib import Path


def encode(text, token_to_id):
    token_ids = []

    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]

        token_ids.append(token_id)

    return token_ids


def decode(token_ids, id_to_token):
    restored = ""

    for token_id in token_ids:
        restored = restored + id_to_token[token_id]

    return restored


token_to_id = {
    "我": 0,
    "爱": 1,
    "学": 2,
    "习": 3,
    "模": 4,
    "型": 5,
    "<unk>": 6,
}
id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]

work_dir = Path.cwd() / "lesson_04_data"
work_dir.mkdir(exist_ok=True)
input_path = work_dir / "input.txt"
output_path = work_dir / "generated_records.json"

# 只有输入文件不存在时，才创建示例；已存在的输入会被保留。
sample_text = '我爱学习\n我爱大模型\n\n学习模型\n'
if not input_path.exists():
    input_path.write_text(sample_text, encoding="utf-8")

raw_text = input_path.read_text(encoding="utf-8")
lines = raw_text.splitlines()

records = []
line_number = 1

for text in lines:
    token_ids = encode(text, token_to_id)
    record = {
        "line_number": line_number,
        "text": text,
        "token_ids": token_ids,
    }
    records.append(record)
    line_number = line_number + 1

# 这是重新生成的结果文件，每次运行都会按当前输入更新。
json_text = json.dumps(records, ensure_ascii=False, indent=2)
output_path.write_text(json_text, encoding="utf-8")

saved_text = output_path.read_text(encoding="utf-8")
loaded_records = json.loads(saved_text)

print("工作目录：", work_dir.resolve())
print("输入文件：", input_path.resolve())
print("结果文件：", output_path.resolve())
print("记录数量：", len(loaded_records))

for record in loaded_records:
    restored = decode(record["token_ids"], id_to_token)
    print("行号：", record["line_number"])
    print("原文：", record["text"])
    print("编号：", record["token_ids"])
    print("解码：", restored)
    print()


工作目录： /workspace/scratch/74d485b2a853/.notebook_build/lesson04_verify_8wzukpot/notebook run/lesson_04_data
输入文件： /workspace/scratch/74d485b2a853/.notebook_build/lesson04_verify_8wzukpot/notebook run/lesson_04_data/input.txt
结果文件： /workspace/scratch/74d485b2a853/.notebook_build/lesson04_verify_8wzukpot/notebook run/lesson_04_data/generated_records.json
记录数量： 4
行号： 1
原文： 我爱学习
编号： [0, 1, 2, 3]
解码： 我爱学习

行号： 2
原文： 我爱大模型
编号： [0, 1, 6, 4, 5]
解码： 我爱<unk>模型

行号： 3
原文： 
编号： []
解码： 

行号： 4
原文： 学习模型
编号： [2, 3, 4, 5]
解码： 学习模型



## 11. 练习区——先停在这里

### 练习 A：修改真实输入文件

1. 找到程序打印的 input.txt，用文本编辑器打开。
2. 在末尾新增一行“我爱模型”，保存为 UTF-8 文本。
3. 重新执行第 10 节完整程序。
4. 核对输入是否保留了你的新增内容，JSON 是否增加了对应记录。
5. 再运行一次，检查程序是否只读取这行，没有自动重复追加它。

如果你使用默认输入，新增一行后应有 5 条记录，最后一行的编号为 `[0, 1, 4, 5]`。


In [12]:
# 在这里记录修改前后的记录数量，以及最后一行的编号。
# 请在文本编辑器中修改 input.txt；本单元不会自动替你修改文件。


### 练习 B：保存解码结果字段

使用两段固定文字“我爱学习”和“我爱大模型”，分别编码、解码。

为每条字典增加 `"decoded_text"` 字段，把解码结果作为这个字段的值。保存到独立的 `exercise_b_records.json`。

写完后，重新读取文件，打印每条记录的原文与解码结果。encode/decode 可使用前面已经定义的版本。


In [13]:
exercise_b_texts = ["我爱学习", "我爱大模型"]
# 创建记录列表，包含 text、token_ids、decoded_text。
# 保存为 lesson_04_data/exercise_b_records.json，再读回来核对。


### 练习 C：进阶——用函数封装保存和读取

编写两个函数：

- `save_records(records, path)`：把数据保存为 JSON，返回保存路径。
- `load_records(path)`：读取 JSON，返回解析后的 Python 数据。

这里的 path 传入 Path 对象。用 `exercise_c_records.json` 验证下面过程：

1. 保存 `[{"text": "我爱", "token_ids": [0, 1]}]`。
2. 把内存变量改成 `[{"text": "学习", "token_ids": [2, 3]}]`，先不保存。
3. 重新读取文件，检查它仍然是上一次保存的内容。
4. 再调用 save_records 保存，重新读取并核对变化。

这个练习帮助你区分：内存变量、输入文件和输出文件。它们只有在你执行对应读取或写入步骤时，才会交换数据。


In [14]:
# 在这里定义 save_records 和 load_records。
# 再按照练习步骤，观察“修改变量”与“重新保存文件”的区别。


## 12. 常见问题

| 现象 | 先检查什么 |
| --- | --- |
| 找不到生成文件 | 看程序打印的绝对路径；当前工作目录不一定是脚本所在目录 |
| 修改输入后结果没有变化 | 是否保存了 input.txt；是否重新执行了读取、处理和保存步骤 |
| Notebook 还显示旧结果 | 是否重新运行了相应单元；输出不会因文件变化而自行刷新 |
| UnicodeDecodeError | 输入文件是否按本课约定保存为 UTF-8 |
| JSONDecodeError | 文件是否仍是合法 JSON；是否误改了引号、逗号或括号 |
| 空格变成未知字符编号 | 本课保留原始空格；当前字符词表没有收录它 |
| 空文件没有记录 | 空文件拆分为 `[]`，因此保存结果也是 `[]` |

下一课会学习怎样在代码中处理常见错误。本课先看懂错误发生在哪一个读取或解析步骤。

## 13. 参考答案——独立尝试后再看

以下参考单元自带所需导入、函数或数据。练习 A 的参考会读取现有输入；B、C 各自使用独立的结果文件。

### 练习 A 参考

这段代码读取当前文件并显示行数和各行内容。新增输入是前面的手工练习步骤；这里不会自动追加文字。


In [15]:
import json
from pathlib import Path

work_dir = Path.cwd() / "lesson_04_data"
work_dir.mkdir(exist_ok=True)
input_path = work_dir / "input.txt"
output_path = work_dir / "generated_records.json"

# 只有输入文件不存在时，才创建示例；已存在的输入会被保留。
sample_text = '我爱学习\n我爱大模型\n\n学习模型\n'
if not input_path.exists():
    input_path.write_text(sample_text, encoding="utf-8")

raw_text = input_path.read_text(encoding="utf-8")
lines = raw_text.splitlines()
print("当前行数：", len(lines))
for line in lines:
    print(line)


当前行数： 4
我爱学习
我爱大模型

学习模型


### 练习 B 参考

解码结果作为普通字符串存入字典，再随整份列表一起保存。默认例子中，第二条记录的 decoded_text 包含 `<unk>`。


In [16]:
import json
from pathlib import Path

def encode(text, token_to_id):
    token_ids = []

    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]

        token_ids.append(token_id)

    return token_ids

def decode(token_ids, id_to_token):
    restored = ""

    for token_id in token_ids:
        restored = restored + id_to_token[token_id]

    return restored

token_to_id = {
    "我": 0,
    "爱": 1,
    "学": 2,
    "习": 3,
    "模": 4,
    "型": 5,
    "<unk>": 6,
}
id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]
work_dir = Path.cwd() / "lesson_04_data"
work_dir.mkdir(exist_ok=True)
exercise_b_path = work_dir / "exercise_b_records.json"
example_texts = ["我爱学习", "我爱大模型"]
example_records = []

for text in example_texts:
    token_ids = encode(text, token_to_id)
    decoded_text = decode(token_ids, id_to_token)
    record = {
        "text": text,
        "token_ids": token_ids,
        "decoded_text": decoded_text,
    }
    example_records.append(record)

example_json = json.dumps(example_records, ensure_ascii=False, indent=2)
exercise_b_path.write_text(example_json, encoding="utf-8")

saved_json = exercise_b_path.read_text(encoding="utf-8")
saved_records = json.loads(saved_json)
print("保存文件：", exercise_b_path.resolve())
for record in saved_records:
    print("原文：", record["text"])
    print("解码：", record["decoded_text"])


保存文件： /workspace/scratch/74d485b2a853/.notebook_build/lesson04_verify_8wzukpot/notebook run/lesson_04_data/exercise_b_records.json
原文： 我爱学习
解码： 我爱学习
原文： 我爱大模型
解码： 我爱<unk>模型


### 练习 C 参考

save_records 返回路径，load_records 返回列表。`saved_records[0]["text"]` 先用 `[0]` 取第一条字典，再用 `["text"]` 取其中的文字。

修改内存变量之后，文件仍保存旧结果；再次调用保存函数后，文件内容才更新。


In [17]:
import json
from pathlib import Path
def save_records(records, path):
    json_text = json.dumps(records, ensure_ascii=False, indent=2)
    path.write_text(json_text, encoding="utf-8")
    return path


def load_records(path):
    json_text = path.read_text(encoding="utf-8")
    return json.loads(json_text)


work_dir = Path.cwd() / "lesson_04_data"
work_dir.mkdir(exist_ok=True)
exercise_c_path = work_dir / "exercise_c_records.json"

example_records = [{"text": "我爱", "token_ids": [0, 1]}]
saved_path = save_records(example_records, exercise_c_path)

example_records = [{"text": "学习", "token_ids": [2, 3]}]
saved_records = load_records(saved_path)
print("当前内存中的文字：", example_records[0]["text"])
print("再次保存前，文件中的文字：", saved_records[0]["text"])

saved_path = save_records(example_records, exercise_c_path)
saved_records = load_records(saved_path)
print("再次保存后，文件中的文字：", saved_records[0]["text"])


当前内存中的文字： 学习
再次保存前，文件中的文字： 我爱
再次保存后，文件中的文字： 学习


## 14. 本课验收

- 能找到程序实际读写的目录，区分工作目录与脚本所在位置。
- 能解释 input.txt 何时创建，以及重新运行时读取什么内容。
- 能把读入的多行字符串拆成列表，说明空行和空文件的区别。
- 能解释一条字典记录与整份记录列表的关系。
- 能区分 dumps/loads、Python 数据/JSON 字符串，以及 UTF-8/中文显示设置。
- 能修改输入文件，重新生成 JSON，并从文件读回结果。
- 能独立完成练习 B；有余力时完成练习 C 的函数封装。

本课完成后，可以发回练习 B 的代码，以及 JSON 中两条记录的内容。下一课学习异常处理与数据检查，让文件不存在、JSON 格式出错等情况更容易定位和处理。

### 官方参考

- [Python：pathlib 路径与文本读写](https://docs.python.org/zh-cn/3/library/pathlib.html)
- [Python：json.dumps 与 json.loads](https://docs.python.org/zh-cn/3/library/json.html)
- [Python：str.splitlines](https://docs.python.org/zh-cn/3/library/stdtypes.html#str.splitlines)
- [Python 教程：输入与输出](https://docs.python.org/zh-cn/3/tutorial/inputoutput.html)
